# Notebook 4 : précision et ordre de convergence

**Question** : de combien l'erreur diminue-t-elle quand on raffine le maillage ?

**Méthode** :
1. un cas dont on connaît la solution exacte (une tache qui se translate),
2. le même calcul sur des maillages de plus en plus fins (dx, dx/2, dx/4, dx/8), **avec CFL constant** (donc dt diminue aussi),
3. l'erreur par rapport à la solution exacte, en normes L1, L2 et L∞,
4. la pente de la courbe log(erreur) en fonction de log(dx) : c'est l'**ordre observé**. On le compare à l'ordre théorique (1 pour l'upwind).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
os.makedirs('figures', exist_ok=True)
from advection import *

Ns = [50, 100, 200, 400]          # nombre de cellules en x : dx = 100/N = 2, 1, 0.5, 0.25

## 1. Translation d'une gaussienne (profil lisse) avec l'upwind

La gaussienne est lisse : c'est le bon profil pour mesurer l'ordre du schéma.

In [ ]:
def etude(profil, schema='upwind', lim='minmod', temps='euler'):
    dxs, err, duree = [], {'L1': [], 'L2': [], 'Linf': []}, []
    for N in Ns:
        r = cas_translation(N, profil, schema, lim, temps)
        dxs.append(r['dx']); duree.append(r['duree'])
        for k in err: err[k].append(r['err'][k])
    return np.array(dxs), {k: np.array(v) for k, v in err.items()}, np.array(duree)

dxs, err, duree = etude('gauss')

print(f"{'dx':>6} {'L1':>11} {'L2':>11} {'Linf':>11} {'ordre L2 local':>15}")
for k in range(len(dxs)):
    loc = np.log(err['L2'][k-1] / err['L2'][k]) / np.log(dxs[k-1] / dxs[k]) if k > 0 else np.nan
    print(f"{dxs[k]:6.2f} {err['L1'][k]:11.3e} {err['L2'][k]:11.3e} {err['Linf'][k]:11.3e} {loc:15.2f}")
for nom in ['L1', 'L2', 'Linf']:
    print(f"Ordre observé ({nom}) : {ordre_observe(dxs, err[nom]):.2f}")

In [ ]:
plt.figure(figsize=(7, 5))
for nom in ['L1', 'L2', 'Linf']:
    plt.loglog(dxs, err[nom], 'o-', label=f"erreur {nom}")
plt.loglog(dxs, err['L2'][0] * (dxs / dxs[0]), 'k--', lw=0.8, label="pente 1")
plt.loglog(dxs, err['L2'][0] * (dxs / dxs[0])**2, 'k:', lw=0.8, label="pente 2")
plt.xlabel("dx"); plt.ylabel("erreur"); plt.title("Upwind + Euler : gaussienne")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/convergence_upwind_gauss.png', dpi=200, bbox_inches='tight')
plt.show()

**À observer** : la pente est proche de 1 (un peu moins aux maillages grossiers, car la gaussienne n'y est représentée que par quelques cellules : on n'est pas encore dans le régime asymptotique). C'est l'ordre théorique de l'upwind, dû à la diffusion numérique ∝ u·dx/2.

## 2. Translation d'un créneau (profil discontinu)

Avec une discontinuité, l'ordre observé chute : la solution n'est pas assez régulière pour que l'ordre théorique s'applique. Pour l'upwind, on attend un ordre de l'ordre de 0,5 en norme L1.

In [ ]:
dxs_c, err_c, duree_c = etude('creneau')
for nom in ['L1', 'L2', 'Linf']:
    print(f"Ordre observé créneau ({nom}) : {ordre_observe(dxs_c, err_c[nom]):.2f}")

plt.figure(figsize=(7, 5))
for nom in ['L1', 'L2']:
    plt.loglog(dxs_c, err_c[nom], 'o-', label=f"créneau {nom}")
plt.loglog(dxs, err['L1'], 's--', label="gaussienne L1")
plt.loglog(dxs_c, err_c['L1'][0] * (dxs_c / dxs_c[0])**0.5, 'k:', lw=0.8, label="pente 1/2")
plt.xlabel("dx"); plt.ylabel("erreur")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/convergence_upwind_creneau.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. Coût de calcul

Quand on divise dx par 2 : 4 fois plus de cellules, et dt est divisé par 2 (CFL constant), donc 2 fois plus de pas. Le coût est multiplié par environ **8** en 2D (en 1/dx³).

In [ ]:
print(f"{'dx':>6} {'temps (s)':>10} {'rapport':>8}")
for k in range(len(dxs)):
    rap = duree[k] / duree[k-1] if k > 0 else np.nan
    print(f"{dxs[k]:6.2f} {duree[k]:10.3f} {rap:8.1f}")

## 4. Rotation solide

Vitesse u = −ω(y − yc), v = ω(x − xc) : le fluide tourne comme un disque rigide. Après **un tour complet**, la tache doit revenir exactement à sa position de départ. Contrairement à la translation, la vitesse varie d'une cellule à l'autre et n'est pas alignée sur la grille : c'est un test beaucoup plus proche du labyrinthe. La divergence discrète de ce champ est exactement nulle.

On compare l'upwind (ordre 1) à un schéma d'ordre 2 (MUSCL avec limiteur de van Leer, détaillé dans le notebook 5).

In [ ]:
cas = {"upwind": dict(schema='upwind', temps='euler'),
       "MUSCL van Leer + RK2": dict(schema='muscl', lim='vanleer', temps='rk2')}
rot = {nom: cas_rotation(N=100, **opt) for nom, opt in cas.items()}

fig, axs = plt.subplots(1, 4, figsize=(16, 4))
r0 = rot["upwind"]
for ax, champ, titre in zip(axs, [r0['c0'], r0['c_exact'], rot["upwind"]['c'], rot["MUSCL van Leer + RK2"]['c']],
                            ["Initial", "Exact après 1 tour", "Upwind", "MUSCL van Leer + RK2"]):
    im = ax.imshow(champ, vmin=0, vmax=1); ax.set_title(titre); ax.axis('off')
plt.colorbar(im, ax=axs, shrink=0.8)
plt.savefig('figures/rotation.png', dpi=200, bbox_inches='tight')
plt.show()

for nom, r in rot.items():
    dm = abs(r['masse'][-1] - r['masse'][0]) / r['masse'][0]
    print(f"{nom:22s} max c = {r['c'].max():.3f}  min c = {r['c'].min():.2e}  erreur L2 = {r['err']['L2']:.3e}  "
          f"variation de masse = {dm:.1e}  temps = {r['duree']:.1f} s")

**À observer** : l'upwind écrase fortement la tache (le maximum chute) et l'étale sur tout l'anneau parcouru ; le schéma d'ordre 2 la conserve beaucoup mieux. La masse est conservée dans les deux cas tant que la tache n'atteint pas les bords du domaine.

In [ ]:
Ns_rot = [50, 100, 200]
plt.figure(figsize=(7, 5))
for nom, opt in cas.items():
    d, e = [], []
    for N in Ns_rot:
        r = cas_rotation(N=N, **opt)
        d.append(r['dx']); e.append(r['err']['L2'])
    d, e = np.array(d), np.array(e)
    print(f"{nom:22s} ordre observé (L2) : {ordre_observe(d, e):.2f}")
    plt.loglog(d, e, 'o-', label=nom)
plt.xlabel("dx"); plt.ylabel("erreur L2"); plt.title("Rotation solide, 1 tour")
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig('figures/convergence_rotation.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Conclusion à rédiger

- Quel ordre observes-tu pour l'upwind sur la gaussienne ? Est-ce cohérent avec la théorie ?
- Pourquoi l'ordre est-il plus faible avec un créneau ?
- Pourquoi l'upwind est-il si mauvais sur la rotation, alors qu'il est correct sur la translation ?
- Combien coûte un maillage deux fois plus fin ?
- Rappel (théorème de Lax) : consistance + stabilité ⇒ convergence. Quel notebook justifie la stabilité, et lequel mesure la convergence ?